# BEWS — Biological Early Warning System for *Aurelia aurita*

Unsupervised two-layer anomaly detector for marine pollution, operating on per-animal behavioural baselines extracted from YOLO-tracked aquarium video.

> **Note** — this notebook expects per-animal feature CSVs (`window_features_<VideoFile>.csv`) for ExpIDs 2–5 plus the experimental database `База_видео_движения.xlsx`. These are not bundled with the repository for size reasons. To run the notebook end-to-end you need to either supply equivalent files of your own or contact the maintainers for the dataset.
>
> If you only have the bundled `data/example/track.csv`, run **`notebooks/quickstart.ipynb`** instead — it demonstrates the same pipeline on a single recording.

This notebook walks through the full pipeline end-to-end:

1. **Load** the experimental database and per-window features from your tracking CSVs
2. **Calibrate** an `AnimalBaseline` per individual (μ, σ) on its own clean-water windows
3. **Calibrate** the detector thresholds (T_z, target, h) on pooled clean-window scores from training animals
4. **Score & alarm** every window with the two-layer detector
5. **Visualise** the results: per-animal time courses, validation table, latency analysis

**Detector architecture in one line:**
> ALARM if  Layer 1: mean|z| > T_z for k consecutive windows  ∨  Layer 2: CUSUM of mean|z| > h

Layer 1 catches step changes and outlier windows; Layer 2 catches sustained small distributional drifts that no single window flags.


## 0 · Setup

The notebook expects:
- `bews.py` next to the notebook (the detector module)
- A folder of per-recording window-feature CSVs, named `window_features_<VideoFile>.csv`
- The experimental database `База_видео_движения.xlsx` with columns ExpID, VideoFile, Time, PollutantType


In [ ]:
# Imports
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure bews.py is importable
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from bews import (AnimalBaseline, BEWSDetector,
                  calibrate_thresholds, load_animal, FEATURES)

# Plotting defaults
plt.rcParams.update({
    'font.size': 9, 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
})
COL = {'clean': '#2E7BB6', 'diesel': '#D9534F'}

print(f'Features used by detector: {FEATURES}')


## 1 · Configure data paths and experimental design

Edit the cell below to match your data layout and the per-animal experimental metadata. Keys:

- `WINDOWS_DIR` — folder with the per-recording CSVs
- `DB_PATH` — path to the experimental database
- `EXPERIMENTS` — for each ExpID, the condition map (clean/diesel per video), known artefact windows, and the injection specification `(video_id, offset_s)` where `offset_s` is the time of injection within that video (negative if injection happened before video start).
- `HELD_OUT_EXPIDS` — list of ExpIDs reserved for testing. Their clean windows do **not** contribute to threshold calibration (T_z, target, h), but their per-animal baselines are still fitted from their own clean data, exactly as they would be in deployment. Use `[]` to calibrate on all animals (no held-out test), `[5]` for a single test fold, or `[3, 5]` for multi-fold validation.


In [ ]:
# === EDIT THIS BLOCK FOR YOUR DATA ===

WINDOWS_DIR = Path('../data')   # <-- edit to point at your folder of window_features_*.csv   # folder with window_features_*.csv
DB_PATH     = Path('../data/experiments.xlsx')   # <-- edit to point at your experimental database

# Per-animal experimental metadata.
# inj_spec=(video_id, offset_s):
#    offset_s>0  -> injection happened that many seconds AFTER video start
#    offset_s<0  -> injection happened that many seconds BEFORE video start
EXPERIMENTS = {
    2: dict(
        cond_map = {'PA170007':'clean','PA170008':'clean',
                    'PA170010':'diesel','PA170011':'diesel'},
        artefact_map = {'PA170008':[0], 'PA170011':[0]},   # camera-motion windows
        inj_spec = ('PA170010', -300),                     # video starts 5 min post-inj
    ),
    3: dict(
        cond_map = {'PA270013':'clean','PA270014':'clean',
                    'PA270015':'diesel','PA270016':'diesel','PA270018':'diesel'},
        artefact_map = {},
        inj_spec = ('PA270015', -30),                      # video starts 30 s post-inj
    ),
    4: dict(
        cond_map = {'PA300020':'clean','PA300021':'clean','PA300022':'clean',
                    'PA300023':'diesel','PA300024':'diesel',
                    'PA300025':'diesel','PA300026':'diesel'},
        artefact_map = {'PA300023':[0]},                   # injection contaminates first window
        inj_spec = ('PA300023', 20),                       # injection at t=20s of this video
    ),
    5: dict(
        cond_map = {'PA300027':'clean','PA300028':'clean',
                    'PA300030':'clean','PA300031':'clean',
                    'PA300032':'diesel','PA300033':'diesel','PA300034':'diesel'},
        artefact_map = {'PA300030':[0], 'PA300031':[0],
                        'PA300032':[0], 'PA300034':[0]},
        inj_spec = ('PA300032', 8),
    ),
}

# Choose which animals are held out from threshold calibration.
# Can be empty list [] (no held-out — use all animals for calibration),
# a single ExpID e.g. [5], or multiple e.g. [4, 5] for multi-fold validation.
HELD_OUT_EXPIDS = [5]

if HELD_OUT_EXPIDS:
    print(f'Configured {len(EXPERIMENTS)} animals; held-out test set = {HELD_OUT_EXPIDS}')
else:
    print(f'Configured {len(EXPERIMENTS)} animals; no held-out — all used for calibration')


## 2 · Assemble the dataset

For each animal we:
1. Read the per-recording window CSVs
2. Tag each window with `cond` (`clean`/`diesel`) and an `artefact` flag
3. Compute an absolute time axis using video timestamps from the database
4. Compute `t_inj_s` = seconds since the pollutant injection (negative for pre-injection windows)


In [ ]:
def assemble_animal(expid: int, spec: dict, db: pd.DataFrame) -> pd.DataFrame:
    """Build a single combined feature table for one animal."""
    sub = db[db.ExpID == expid].copy()
    sub['Time'] = pd.to_datetime(sub['Time'].astype(str), format='mixed', errors='coerce')
    t_ref = sub['Time'].min()

    video_starts = {r['VideoFile']: (r['Time'] - t_ref).total_seconds()
                    for _, r in sub.iterrows()
                    if r['VideoFile'] in spec['cond_map']}

    inj_abs = None
    if spec['inj_spec'] is not None:
        vid, off = spec['inj_spec']
        if vid in video_starts:
            inj_abs = video_starts[vid] + off

    paths = {v: str(WINDOWS_DIR / f'window_features_{v}.csv')
             for v in spec['cond_map']}
    df = load_animal(paths,
                     cond_map=spec['cond_map'],
                     artefact_map=spec['artefact_map'],
                     inj_abs_s=inj_abs,
                     video_starts_s=video_starts)
    df['expid'] = expid
    return df

db = pd.read_excel(DB_PATH)
animals = {expid: assemble_animal(expid, spec, db)
           for expid, spec in EXPERIMENTS.items()}

# Quick summary
print(f'{"ExpID":>5} {"clean (ok)":>11} {"diesel (ok)":>12} {"artefact":>9}')
for expid, df in animals.items():
    ok = ~df.artefact
    nc = ((df.cond == 'clean')  & ok).sum()
    nd = ((df.cond == 'diesel') & ok).sum()
    na = df.artefact.sum()
    print(f'{expid:>5} {nc:>11} {nd:>12} {na:>9}')


## 3 · Fit per-animal baselines

Each animal gets its own `AnimalBaseline`: per-feature μ and σ. After this step we score every window — clean and diesel — against the baseline of *its own animal*.

We split the scoring stream by condition so that the CUSUM layer (which has memory) treats clean and diesel as independent sessions.


In [ ]:
SCORE_COLS = ['video','cond','t_mid','abs_t','t_inj_s']

baselines, scored = {}, {}
for expid, df in animals.items():
    df_ok = df[~df.artefact].copy()
    clean = df_ok[df_ok.cond == 'clean']
    if len(clean) < 10:
        print(f'⚠️  ExpID {expid}: only {len(clean)} clean windows — baseline may be unstable.')

    bl = AnimalBaseline().fit(clean)
    baselines[expid] = bl

    # Score per condition (so CUSUM resets between sessions)
    scored[expid] = {}
    for cond in ['clean', 'diesel']:
        sub = df_ok[df_ok.cond == cond].sort_values('abs_t').reset_index(drop=True)
        if len(sub) == 0:
            continue
        sc = bl.score(sub)
        ctx = sub[[c for c in SCORE_COLS if c in sub.columns]].reset_index(drop=True)
        scored[expid][cond] = pd.concat([ctx, sc.reset_index(drop=True)], axis=1)
        scored[expid][cond]['expid'] = expid

    print(f'ExpID {expid}: baseline fitted on {len(clean)} clean windows; '
          f"clean μ(Fp)={bl.mu['Fp']:.3f}  σ(Fp)={bl.sd['Fp']:.3f}")


## 4 · Calibrate detector thresholds on training animals only

We pool the clean-window scores of every animal *except* the held-out one and pick:
- **T_z** = 99th percentile of pooled `mean_abs_z` → ~1 % per-window FPR on Layer 1
- **CUSUM target** = median pooled `mean_abs_z`
- **CUSUM h** = `safety_factor` × max CUSUM trace observed on any training-animal clean stream

The k-consecutive rule on Layer 1 then drives the session-level FPR several orders of magnitude lower.


In [ ]:
train_expids = [e for e in scored if e not in HELD_OUT_EXPIDS]
train_clean_scores = [scored[e]['clean'] for e in train_expids
                      if 'clean' in scored[e]]

if not train_clean_scores:
    raise RuntimeError(
        'No training animals available — every configured ExpID is in HELD_OUT_EXPIDS. '
        'Threshold calibration requires at least one non-held-out animal with clean windows.')

cal = calibrate_thresholds(train_clean_scores,
                           target_fpr_per_window=0.01,
                           cusum_safety_factor=1.3)

print(f'Training animals : {train_expids}')
print(f'Held-out animals : {HELD_OUT_EXPIDS}')
print(f'Calibration windows: {cal["calibration_n"]}')
print()
print(f'  T_z   (Layer 1 threshold)         = {cal["threshold_mean_abs_z"]:.3f}')
print(f'  target (CUSUM reference)          = {cal["cusum_target"]:.3f}')
print(f'  h     (Layer 2 / CUSUM threshold) = {cal["cusum_h"]:.3f}')
print(f'  max clean CUSUM observed in train = {cal["cusum_max_clean"]:.3f}')


## 5 · Run the two-layer detector on every window

The detector adds the following columns:
- `above_z` — Layer 1 above threshold (single window)
- `alarm_fast` — Layer 1 met for k consecutive windows
- `alarm_slow` — Layer 2 (CUSUM) crossed h
- `alarm` — final OR of fast and slow

`cusum_S` shows the running CUSUM trace for diagnostics.


In [ ]:
K_CONSECUTIVE = 3

detector = BEWSDetector(
    threshold_mean_abs_z = cal['threshold_mean_abs_z'],
    cusum_target         = cal['cusum_target'],
    cusum_h              = cal['cusum_h'],
    k_consecutive        = K_CONSECUTIVE,
)

alarmed = {}
for expid, by_cond in scored.items():
    alarmed[expid] = {}
    for cond, sc in by_cond.items():
        alarmed[expid][cond] = detector.apply(sc).reset_index(drop=True)

# Peek at one held-out (or first) animal's diesel response, window-by-window
view_cols = ['t_inj_s','mean_abs_z','cusum_S',
             'above_z','alarm_fast','alarm_slow','alarm']
peek_expid = HELD_OUT_EXPIDS[0] if HELD_OUT_EXPIDS else sorted(alarmed)[0]
print(f'Window-by-window detector trace for ExpID {peek_expid} (diesel phase):')
held = alarmed[peek_expid]['diesel'][view_cols].round(3)
held.head(15)


## 6 · Validation summary

For every animal we report:
- **clean FPR** — fraction of clean windows that raise an alarm (must be ~0)
- **diesel detection** — yes/no at the animal level
- **latency** — minutes between injection and the first alarm
- **layer breakdown** — which layer fired

A working detector should show 0 false alarms on every animal's clean period and detect all diesel exposures.


In [ ]:
rows = []
for expid in sorted(alarmed):
    for cond in ['clean', 'diesel']:
        if cond not in alarmed[expid]: continue
        sub = alarmed[expid][cond]
        n  = len(sub)
        na = int(sub['alarm'].sum())
        first_t = (sub.loc[sub['alarm'].idxmax(), 't_inj_s'] / 60
                   if na > 0 and 't_inj_s' in sub.columns else np.nan)
        rows.append(dict(
            ExpID = expid,
            role  = 'TEST' if expid in HELD_OUT_EXPIDS else 'train',
            phase = cond,
            n_windows  = n,
            n_alarms   = na,
            alarm_rate = round(100 * na / max(n, 1), 1),
            first_alarm_min = round(first_t, 2) if pd.notna(first_t) else None,
            layer1_alarms = int(sub['alarm_fast'].sum()),
            layer2_alarms = int(sub['alarm_slow'].sum()),
        ))

summary = pd.DataFrame(rows)
summary


## 7 · Visualise per-animal time courses

Each panel shows mean|z| (Layer 1 score) on a time axis aligned to injection.
Clean windows are shown on the left of zero, diesel windows on the right.
The dotted line is the calibrated T_z. Gold stars mark windows at which the full two-layer detector raises an alarm.


In [ ]:
def plot_animal_panel(ax, alarmed_animal: dict, expid: int,
                       T_z: float, role: str = ''):
    clean  = alarmed_animal.get('clean')
    diesel = alarmed_animal.get('diesel')

    if diesel is not None and 't_inj_s' in diesel.columns and diesel['t_inj_s'].notna().any():
        diesel_x = diesel['t_inj_s'].values / 60
    else:
        diesel_x = np.array([])

    if clean is not None and len(clean):
        clean_x = np.arange(len(clean)) * 0.5
        clean_x = clean_x - clean_x.max() - 1
    else:
        clean_x = np.array([])

    ax.axhspan(0, T_z, color='#EEF4FA', alpha=0.6, zorder=0)
    ax.axhline(T_z, ls=':', color='k', lw=1, alpha=0.6,
               label=f'T_z = {T_z:.2f}')
    ax.axvline(0, color='k', ls='--', lw=1.2, alpha=0.7)

    if len(clean_x):
        ax.plot(clean_x, clean['mean_abs_z'], '-', color=COL['clean'], lw=1, alpha=0.6)
        ax.scatter(clean_x, clean['mean_abs_z'], s=14, color=COL['clean'],
                   alpha=0.8, edgecolor='white', lw=0.4, label='clean', zorder=3)

    if len(diesel_x):
        ax.plot(diesel_x, diesel['mean_abs_z'], '-', color=COL['diesel'], lw=1, alpha=0.6)
        ax.scatter(diesel_x, diesel['mean_abs_z'], s=14, color=COL['diesel'],
                   alpha=0.85, edgecolor='white', lw=0.4, label='diesel', zorder=3)

    for x, sub in [(clean_x, clean), (diesel_x, diesel)]:
        if sub is None or len(sub) == 0 or len(x) == 0: continue
        m = sub['alarm'].values
        if m.any():
            existing = ax.get_legend_handles_labels()[1]
            ax.scatter(x[m], sub.loc[m, 'mean_abs_z'],
                       s=90, marker='*', color='#FFD700', edgecolor='k', lw=0.8,
                       zorder=5,
                       label='ALARM' if 'ALARM' not in existing else None)

    if diesel is not None and diesel['alarm'].any() and len(diesel_x):
        i0 = diesel['alarm'].idxmax()
        t0 = diesel.loc[i0, 't_inj_s'] / 60
        ymax = diesel['mean_abs_z'].max()
        ax.annotate(f'first alarm: +{t0:.1f} min',
                    xy=(t0, 0.3), xytext=(t0 + 1, max(0.1, ymax*0.05)),
                    fontsize=8.5, color='#8B4513',
                    arrowprops=dict(arrowstyle='->', color='#8B4513', lw=0.8))

    ymax = max(2, np.nanmax(np.r_[
        diesel['mean_abs_z'].values if diesel is not None else [0],
        clean['mean_abs_z'].values  if clean  is not None else [0]]) * 1.05)
    ax.set_ylim(0, ymax)
    ax.set_ylabel('mean |z|')
    ax.grid(axis='y', alpha=0.25, lw=0.5)

    fa_d = int(diesel['alarm'].sum()) if diesel is not None else 0
    nd   = len(diesel) if diesel is not None else 0
    fa_c = int(clean['alarm'].sum())  if clean  is not None else 0
    nc   = len(clean)  if clean  is not None else 0
    title = f'ExpID {expid}'
    if role:
        title += f'  ({role})'
    ax.text(0.012, 0.94, title, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')
    ax.text(0.012, 0.84,
            f'clean: {fa_c}/{nc} alarms · diesel: {fa_d}/{nd} alarms',
            transform=ax.transAxes, fontsize=8.3, color='0.3', va='top')

expids = sorted(alarmed)
fig, axes = plt.subplots(len(expids), 1, figsize=(8.5, 2.0*len(expids)),
                          gridspec_kw=dict(hspace=0.35))
if len(expids) == 1: axes = [axes]
for ax, expid in zip(axes, expids):
    role = 'TEST' if expid in HELD_OUT_EXPIDS else 'train'
    plot_animal_panel(ax, alarmed[expid], expid,
                      T_z=cal['threshold_mean_abs_z'], role=role)
axes[0].legend(loc='upper right', fontsize=7.8, framealpha=0.92, ncol=2)
axes[-1].set_xlabel('time since injection (min)   [clean windows compressed on the left]')
fig.suptitle('BEWS validation across animals', fontsize=11, y=0.995)
plt.show()


## 8 · Two-layer detail on the held-out animals

The two layers operate on the same scalar (mean|z|) but extract different aspects of anomaly:
- Layer 1 — spikes hard on step changes; threshold-and-confirm logic
- Layer 2 (CUSUM) — accumulates small persistent shifts; useful when mean|z| stays modest but consistently above target

If multiple animals are held out, one figure per animal is produced.
If `HELD_OUT_EXPIDS` is empty, the first animal in the dataset is shown for illustration.


In [ ]:
def plot_two_layer_detail(target_expid: int):
    """Two-layer detail panel for one animal; returns the figure."""
    clean  = alarmed[target_expid].get('clean')
    diesel = alarmed[target_expid].get('diesel')
    if clean is None or diesel is None:
        print(f'ExpID {target_expid}: missing clean or diesel data, skipping')
        return None

    if len(clean):
        clean_x = np.arange(len(clean)) * 0.5
        clean_x = clean_x - clean_x.max() - 1
    else:
        clean_x = np.array([])
    diesel_x = (diesel['t_inj_s'].values / 60
                if 't_inj_s' in diesel.columns else np.arange(len(diesel)) * 0.5)

    fig, axes = plt.subplots(2, 1, figsize=(8.5, 4.4), sharex=True,
                              gridspec_kw=dict(hspace=0.12))
    layers = [
        (axes[0], 'mean_abs_z', cal['threshold_mean_abs_z'], 'Layer 1\nmean |z|'),
        (axes[1], 'cusum_S',    cal['cusum_h'],              'Layer 2\nCUSUM S'),
    ]
    for ax, col, thr, ylab in layers:
        ax.axhline(thr, ls=':', color='k', lw=1, alpha=0.6)
        if len(clean):
            ax.plot(clean_x,  clean[col],  '-o', color=COL['clean'],
                    ms=3.5, lw=1, alpha=0.7)
        if len(diesel):
            ax.plot(diesel_x, diesel[col], '-o', color=COL['diesel'],
                    ms=3.5, lw=1)
        ax.axvline(0, color='k', ls='--', lw=1.2, alpha=0.7)
        ax.set_ylabel(ylab)
        ax.grid(alpha=0.25, lw=0.5)
        ax.text(0.99, 0.95, f'threshold {thr:.2f}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color='0.3')

    if diesel['alarm'].any():
        first_t = diesel.loc[diesel['alarm'].idxmax(), 't_inj_s'] / 60
        for ax in axes:
            ax.axvspan(first_t-0.25, first_t+0.25, color='#FFD700', alpha=0.25)
        axes[0].text(first_t, axes[0].get_ylim()[1]*0.92,
                     f'FIRST ALARM\n+{first_t:.1f} min',
                     ha='center', fontsize=8.5, color='#8B4513', fontweight='bold')
    else:
        axes[0].text(0.5, 0.92, 'NO ALARM RAISED', transform=axes[0].transAxes,
                     ha='center', fontsize=9.5, color='#8B0000', fontweight='bold')

    axes[-1].set_xlabel('time since injection (min)')
    role = 'held-out' if target_expid in HELD_OUT_EXPIDS else 'training'
    axes[0].set_title(f'ExpID {target_expid} ({role}) — two-layer detector traces',
                      fontsize=10.5, loc='left')
    return fig

# Render one figure per held-out animal; fall back to first animal if none held out
detail_targets = HELD_OUT_EXPIDS if HELD_OUT_EXPIDS else [sorted(alarmed)[0]]
for expid in detail_targets:
    fig = plot_two_layer_detail(expid)
    if fig is not None:
        plt.show()


## 9 · Diagnostics: clean vs diesel mean|z| distributions

A useful sanity check before deploying the detector on a new animal: compare the distribution of `mean|z|` between the animal's own clean and diesel windows. Heavy overlap means Layer 1 alone won't separate them — the detector will rely on Layer 2 (CUSUM) for that animal.


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.6))

pos = 0
xticks, xlabels = [], []
for expid in sorted(alarmed):
    for cond in ['clean', 'diesel']:
        sub = alarmed[expid].get(cond)
        if sub is None or not len(sub): continue
        vals = sub['mean_abs_z'].dropna().values
        ax.boxplot(vals, positions=[pos], widths=0.55, patch_artist=True,
                   medianprops=dict(color='k', lw=1.1),
                   flierprops=dict(marker='o', ms=2.5, alpha=0.4),
                   boxprops=dict(facecolor=COL[cond], alpha=0.35,
                                 edgecolor=COL[cond]),
                   whiskerprops=dict(color=COL[cond]),
                   capprops=dict(color=COL[cond]))
        xticks.append(pos); xlabels.append(f'E{expid}\n{cond[:3]}')
        pos += 1
    pos += 0.4

ax.axhline(cal['threshold_mean_abs_z'], ls=':', color='k', lw=1, alpha=0.7,
           label=f"T_z = {cal['threshold_mean_abs_z']:.2f}")
ax.axhline(cal['cusum_target'], ls='--', color='0.4', lw=1, alpha=0.7,
           label=f"CUSUM target = {cal['cusum_target']:.2f}")
ax.set_xticks(xticks); ax.set_xticklabels(xlabels, fontsize=7.5)
ax.set_ylabel('mean |z|')
ax.set_title('Clean vs diesel mean|z| per animal', fontsize=10)
ax.grid(axis='y', alpha=0.25, lw=0.5)
ax.legend(fontsize=8.5, loc='upper left')

plt.tight_layout()
plt.show()


## 10 · Save outputs

Persist the scored windows and the validation summary so they can be reused without re-running the pipeline.


In [ ]:
OUT = Path('./bews_outputs'); OUT.mkdir(exist_ok=True)

all_scored = []
for expid, by_cond in alarmed.items():
    for cond, df in by_cond.items():
        all_scored.append(df)
pd.concat(all_scored, ignore_index=True).to_csv(OUT / 'bews_scored_all.csv', index=False)

summary.to_csv(OUT / 'bews_validation_summary.csv', index=False)
pd.Series(cal).to_csv(OUT / 'bews_calibration.csv', header=['value'])

print(f'Saved to {OUT.resolve()}/')
for f in sorted(OUT.iterdir()): print(f'  {f.name}')


## How to use this on a new animal

When a new individual arrives in the monitoring aquarium:

1. **Record ≥15 min of clean-water video.** Any less and the per-animal σ will be poorly estimated and z-scores become unreliable.
2. **Process the clean recording** through the YOLO + feature-extraction pipeline to produce a `window_features_*.csv`.
3. **Fit only the `AnimalBaseline`** for this animal — you do *not* recalibrate the global thresholds (T_z, target, h). Those are deployment-fixed and shipped with the system.
4. **Stream new windows through `detector.apply(...)`.** The output `alarm` column is the live BEWS signal.

The minimal code path for runtime monitoring is:

```python
bl = AnimalBaseline().fit(clean_calibration_windows)
detector = BEWSDetector(**deployment_thresholds)   # T_z, target, h fixed
result = detector.apply(bl.score(new_window_batch))
if result['alarm'].any():
    raise_alert()
```

That is the entire detector at deployment — under 10 lines of Python around `bews.py`, with no machine-learning dependencies.
